<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Preprocesamiento y Data Augmentation
**Proyecto:** Detección de Deslizamientos mediante IA | **EAFIT** 
**Objetivo:** Definir la pipeline de normalización, normalización Z-score y aumentación geométrica para datos de 14 canales.

In [ ]:
import os, sys, h5py, torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ── CONFIGURACIÓN DE RUTAS ──────────────────────────────────────────────────
DATA_ROOT = Path('/content/landslide4sense')

if not (DATA_ROOT / 'TrainData').exists():
    print('Dataset no detectado. Descargando...')
    !kaggle datasets download -d tekbahadurkshetri/landslide4sense -p /content/ --unzip
    if os.path.exists('/content/TrainData'):
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        import shutil
        for d in ['TrainData', 'ValidData']: shutil.move(f'/content/{d}', DATA_ROOT / d)

img_dir = DATA_ROOT / "TrainData" / "img"
mask_dir = DATA_ROOT / "TrainData" / "mask"
img_list = sorted(list(img_dir.glob("*.h5")))
mask_list = sorted(list(mask_dir.glob("*.h5")))

CHANNEL_NAMES = ['SAR VV', 'SAR VH', 'SAR VV/VH', 'B2-Blue', 'B3-Green', 'B4-Red', 
                 'B5-RE1', 'B6-RE2', 'B7-RE3', 'B8-NIR', 'B8A-RE4', 'B11-SWIR1', 'B12-SWIR2', 'DEM']
print(f"✓ Dataset listo: {len(img_list)} muestras detectadas.")

## 1. Pipeline de Normalización
Comparación visual entre datos brutos, Min-Max y Z-Score (Estandarización).

In [ ]:
def zscore_norm(patch):
    # Normalización por canal (media=0, std=1)
    mean = np.mean(patch, axis=(0, 1))
    std = np.std(patch, axis=(0, 1)) + 1e-6
    return (patch - mean) / std

def minmax_norm(patch):
    pmin = np.min(patch, axis=(0, 1))
    pmax = np.max(patch, axis=(0, 1)) + 1e-6
    return (patch - pmin) / (pmax - pmin)

# Cargar un parche de ejemplo con deslizamiento
sample_idx = 1816 # Ajustado según tu error previo
with h5py.File(img_list[0], 'r') as hf: patch = hf[list(hf.keys())[0]][()]

patch_z = zscore_norm(patch)
patch_mm = minmax_norm(patch)

# Visualización de Histogramas
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
channels_to_plot = [5, 9, 13] # Red, NIR, DEM

for i, (data, title) in enumerate(zip([patch, patch_mm, patch_z], ['Bruto', 'Min-Max', 'Z-Score'])):
    for ch in channels_to_plot:
        sns.kdeplot(data[:,:,ch].flatten(), ax=axes[i], label=CHANNEL_NAMES[ch], fill=True, alpha=0.2)
    axes[i].set_title(f"Distribución: {title}")
    axes[i].legend()

plt.tight_layout()
plt.show()

## 2. Definición del Dataset y Augmentation
Implementamos rotaciones y flips aleatorios para mejorar la generalización del modelo.

In [ ]:
class LandslideDataset(Dataset):
    def __init__(self, img_files, mask_files, augment=False):
        self.img_files = img_files
        self.mask_files = mask_files
        self.augment = augment

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        with h5py.File(self.img_files[idx], 'r') as hf: 
            img = hf[list(hf.keys())[0]][()]
        with h5py.File(self.mask_files[idx], 'r') as hf: 
            mask = hf[list(hf.keys())[0]][()]
        
        img = zscore_norm(img)
        
        if self.augment:
            # Rotación aleatoria (90, 180, 270)
            k = np.random.randint(0, 4)
            img = np.rot90(img, k)
            mask = np.rot90(mask, k)
            # Flip horizontal
            if np.random.random() > 0.5:
                img = np.flip(img, axis=1).copy()
                mask = np.flip(mask, axis=1).copy()

        img = torch.from_numpy(img.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask).long()
        return img, mask

print("✓ Clase Dataset definida con Data Augmentation integrado.")

## 3. Verificación de Aumentación
Visualización de parches transformados.

In [ ]:
ds = LandslideDataset(img_list, mask_list, augment=True)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for i in range(4):
    img, mask = ds[1816] # Misma muestra, diferentes aumentaciones
    # Para visualización usamos las bandas RGB [5, 4, 3] normalizadas 0-1
    rgb = img[[5, 4, 3]].numpy().transpose(1, 2, 0)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
    
    axes[0, i].imshow(rgb)
    axes[0, i].set_title(f"Augmentación {i+1}")
    axes[1, i].imshow(mask, cmap='Reds')
    axes[0, i].axis('off'); axes[1, i].axis('off')

plt.suptitle("Validación de Augmentation Geométrica", fontsize=16)
plt.show()